# Okey Vision TS - YOLOv8 Production MLOps Training Pipeline
This notebook guides you through hardware verification, library pinning, Weights & Biases experiment tracking, downloading **both Roboflow and Kaggle** datasets (using Roboflow API and `kagglehub`), **parsing VGG Image Annotator (VIA) JSON coordinates to YOLO format**, aligning classes, training the YOLOv8 model, validation, and ONNX half-precision (FP16) WebAssembly export.

### Step 1: Pin Dependencies & Verify GPU

In [1]:
!nvidia-smi
!pip install onnxscript
# Pinning stable dependency versions for reproducibility
!pip install ultralytics==8.2.0 roboflow kagglehub onnx wandb opencv-python pyyaml

Wed Jul 15 10:02:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Weights & Biases (W&B) Experiment Tracking Setup

In [3]:
import wandb
# Connect to W&B to track training logs, metrics, and GPU usages
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: atacanymc (atacanymc-anadolu-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### Step 3: Download Dataset (Kaggle)

We use the official `kagglehub` library to fetch the latest version of the Kaggle Okey dataset without manual API key copying.

In [4]:
import kagglehub

# Download latest version
output_dir = '/content/drive/MyDrive/Kaggle/atacanyaymac/okey-data'
path = kagglehub.dataset_download(handle="atacanyaymac/okey-data", output_dir=output_dir, force_download=True)
print("Path to dataset files:", path)

roboflow_versions_path = f"/content/drive/MyDrive{path}/okey-data/roboflow-versions"
dataset_mrk_1_path = f"{roboflow_versions_path}/Rummikub-1/data.yaml"
dataset_mrk_2_path = f"{roboflow_versions_path}/Rummikub-2/data.yaml"
dataset_mrk_3_path = f"{roboflow_versions_path}/Rummikub-3/data.yaml"

print("Path to Rummikub-1 dataset:", dataset_mrk_1_path)
print("Path to Rummikub-2 dataset:", dataset_mrk_2_path)
print("Path to Rummikub-3 dataset:", dataset_mrk_3_path)

Using Colab cache for faster access to the 'okey-data' dataset.
Path to dataset files: /kaggle/input/okey-data
Path to Rummikub-1 dataset: /content/drive/MyDrive/kaggle/input/okey-data/okey-data/roboflow-versions/Rummikub-1/data.yaml
Path to Rummikub-2 dataset: /content/drive/MyDrive/kaggle/input/okey-data/okey-data/roboflow-versions/Rummikub-2/data.yaml
Path to Rummikub-3 dataset: /content/drive/MyDrive/kaggle/input/okey-data/okey-data/roboflow-versions/Rummikub-3/data.yaml


### Step 5: YOLOv8 Training with Custom Okey Augmentations

We configure:
- `hsv_v=0.5`: Simulates lighting glares and reflection on plastic tiles
- `degrees=10`, `perspective=0.0001`: Simulates looking at the okey rack from different angles
- `deterministic=True`: Guarantees reproducible training runs
- `patience=15`: Activates Early Stopping to prevent overfitting

*(Note: We patch `torch.load` to default `weights_only=False` to prevent PyTorch 2.6+ unpickling errors for custom Ultralytics models)*

In [ ]:
import wandb
from ultralytics import YOLO
import functools
import torch

torch.load = functools.partial(torch.load, weights_only=False)

# 1. WandB'ye mevcut Run ID ile bağlanın (Grafiklerin kopmaması için)
# WandB panelinize gidin, yarıda kesilen run'a tıklayın.
# URL'de veya sol üstte yazan 8 haneli ID'yi (örneğin: 5lu3tv8a) buraya yapıştırın.
run = wandb.init(
    project="OkeySolver_runs",
    id="BURAYA_WANDB_RUN_ID_YAZIN",
    resume="must" # W&B'ye bu eğitimin sıfırdan başlamadığını, devam ettiğini söyler
)

# 2. Drive'da güvenle saklanan en son ağırlığı (last.pt) yükleyin
# best.pt DEĞİL, momentum ve optimizer durumunu tutan last.pt yüklenmelidir!
model = YOLO("/content/drive/MyDrive/OkeySolver_runs/run_mark5_gpu_optimized/weights/last.pt")

# 3. Sadece resume=True diyerek eğitimi tetikleyin
# DİKKAT: epochs, imgsz, batch gibi parametreleri TEKRAR YAZMIYORSUNUZ!
# YOLO tüm bu ayarları last.pt dosyasının içindeki hafızadan kendi okur.
results = model.train(resume=True)

wandb.finish()

In [ ]:
import os
import torch
import kagglehub
import wandb
from ultralytics import YOLO
import functools

# PyTorch güvenlik yamasını uygula
torch.load = functools.partial(torch.load, weights_only=False)

# 1. GPU Kontrolü (Hata yapmanızı engeller)
if not torch.cuda.is_available():
    raise SystemError("GPU AKTİF DEĞİL! Lütfen Colab menüsünden Runtime -> Change runtime type -> T4 GPU seçin.")
training_device = 0

# 2. HIZLI G/Ç İÇİN: Veriyi Colab Yerel Diskine İndir (Drive'a DEĞİL)
local_dataset_dir = '/content/okey_dataset'
path = kagglehub.dataset_download(handle="atacanyaymac/okey-data", output_dir=local_dataset_dir, force_download=True)

# data.yaml yolunu belirle
dataset_mrk_2_path = os.path.join(path, "okey-data", "roboflow-versions", "Rummikub-2", "data.yaml")

# 3. GÜVENLİK İÇİN: Eğitim Sonuçlarını Google Drive'a Kaydet
# Böylece Colab çökse bile ağırlıklarınız (best.pt) Drive'da güvende olur
drive_project_path = "/content/drive/MyDrive/OkeySolver_runs"
os.makedirs(drive_project_path, exist_ok=True)

# 4. Modeli Yükle (Eğer sıfırdan eğitiyorsanız 'yolov8n.pt' kullanın)
# Eğer kendi modelinizin üstüne eğitecekseniz: YOLO("/content/drive/MyDrive/.../best.pt")
model = YOLO('yolov8n.pt')
run_name = "run_mark6_300_epochs"

# 5. WandB Başlat
run = wandb.init(
    project="OkeySolver_runs",
    name=run_name,
    job_type="training"
)

# 6. OPTİMİZE EDİLMİŞ EĞİTİM (Tek seferde 50 Epoch)
results = model.train(
    data=dataset_mrk_2_path,
    epochs=300,                 # Eskisi gibi 10+10 yapmayın.
    imgsz=640,
    batch=-1,                   # AutoBatch: T4 GPU'nun RAM'ine en uygun batch boyutunu otomatik bulur (genelde 16 veya 32)
    device=training_device,
    project=drive_project_path, # Sonuçlar doğrudan Google Drive'a yazılır!
    name=run_name,
    workers=4,                  # Colab için en ideal dataloader işçi sayısı
    cache=True,                 # Veriyi Colab RAM'ine alır, epoch geçişleri ışık hızında olur!
    # Augmentations (Sizin ayarlarınız)
    hsv_v=0.5,
    degrees=10,
    perspective=0.0001,
    deterministic=True,
    patience=15                 # Overfit olursa 15 epoch bekle ve erken durdur
)

# WandB'yi Kapat
wandb.finish()

Using Colab cache for faster access to the 'okey-data' dataset.


New https://pypi.org/project/ultralytics/8.4.95 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/kaggle/input/okey-data/okey-data/roboflow-versions/Rummikub-2/data.yaml, epochs=300, time=None, patience=15, batch=-1, imgsz=640, save=True, save_period=-1, cache=True, device=0, workers=4, project=/content/drive/MyDrive/OkeySolver_runs, name=run_mark6_300_epochs, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=F

train: Scanning /kaggle/input/okey-data/okey-data/roboflow-versions/Rummikub-2/train/labels... 1371 images, 3 backgrounds, 0 corrupt: 100%|██████████| 1371/1371 [00:08<00:00, 162.23it/s]


train: WARNING ⚠️ Cache directory /kaggle/input/okey-data/okey-data/roboflow-versions/Rummikub-2/train is not writeable, cache not saved.
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 13017, len(boxes) = 13182. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.


train: Caching images (1.6GB RAM): 100%|██████████| 1371/1371 [00:06<00:00, 212.79it/s]

...

     38/300      2.45G       0.77     0.9799      1.022         10        640: 100%|██████████| 138/138 [00:39<00:00,  3.50it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:01<00:00,  3.48it/s]


                   all        107       1179      0.628      0.695      0.717      0.499

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     39/300      2.43G      0.774     0.9746      1.023          6        640: 100%|██████████| 138/138 [00:38<00:00,  3.57it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:01<00:00,  5.70it/s]


                   all        107       1179      0.629      0.713      0.731      0.511

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     40/300      2.49G     0.7746     0.9472      1.007         98        640:  86%|████████▌ | 118/138 [00:34<00:04,  4.22it/s]

### Step 7: Export to ONNX with FP16 Quantization

We use `half=True` (FP16) which reduces model footprint by **~50%** without noticeable accuracy loss, providing lightning-fast loading speeds on GitHub Pages.

In [ ]:
from ultralytics import YOLO
import torch
import functools
import os

# Patch torch.load to default weights_only=False to allow unpickling custom YOLO modules in PyTorch 2.6+
torch.load = functools.partial(torch.load, weights_only=False)

# Construct the path to the best.pt from the most recent training run
# output_dir is from cell y7c-6_mG785A, and the save_dir for training is f"{output_dir}/runs/model-III"
# The weights are typically stored in a 'weights' subfolder within the run directory
model_path = os.path.join(artifact_dir, "best.pt")

# Load the model
model = YOLO(model_path)

# Export to ONNX with half-precision floating point enabled using the Python API
model.export(format="onnx", opset=12, imgsz=640)